# PATSTAT Hit Probability — citation percentile within WIPO sector x filing-year cohorts

The twin of `PatentView/notebook/patent_hit_probability.ipynb`. Within each (`wipo_sector`, `filing_year`)
cohort, `pctl_<col>` is the citation percentile in [0, 1] of every count column in `patstat_citation.parquet`
(`rank(method='min') / n`, so zero-citation ties share the lowest rank). Read hits as top 1 % <=> `pctl >= 0.99`,
top 5 % <=> `>= 0.95`. Legacy aliases `pctl_c3/c5/c10/call` (= `pctl_C_3/…`) are kept for the readers that
use the short names.

Cohort sector = `patstat_metadata.wipo_sector` (from the WIPO field with the largest weight in
`tls230_appln_techn_field`); applications without a technology field are not ranked.

## Output
`PATSTAT/output/patstat_hit_probability.parquet` — `appln_id, wipo_sector, filing_year` + `pctl_<col>` for every
`C*`, `uniqueC*` column, + the four aliases.

In [ ]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
OUT_FP = ps.out('patstat_hit_probability.parquet')
ps.preflight('patstat_hit_probability')

CIT, META = ps.out('patstat_citation.parquet'), ps.out('patstat_metadata.parquet')
COUNT_COLS = [c for c in pq.ParquetFile(CIT).schema_arrow.names if c != 'appln_id']
LEGACY = {'pctl_c3': 'C_3', 'pctl_c5': 'C_5', 'pctl_c10': 'C_10', 'pctl_call': 'C_all'}
print(f'{len(COUNT_COLS)} citation columns to rank')
con = ps.connect()

## 1. Percentiles within (sector, filing year)

In [ ]:
%%time
# rank(method='min', pct=True) == rank() / count(*) within the cohort; every column in one window pass.
ranks = ', '.join(f"(rank() OVER (PARTITION BY m.wipo_sector, m.filing_year ORDER BY coalesce(c.{col}, 0)) "
                  f"/ count(*) OVER (PARTITION BY m.wipo_sector, m.filing_year))::FLOAT AS pctl_{col}" for col in COUNT_COLS)
con.execute(f"""COPY (
  SELECT m.appln_id, m.wipo_sector, m.filing_year, {ranks}
  FROM read_parquet('{META}') m LEFT JOIN read_parquet('{CIT}') c USING (appln_id)
  WHERE m.wipo_sector IS NOT NULL
) TO '{OUT_FP}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
# the four short aliases, appended in a second pass so the window query stays one statement
con.execute(f"""COPY (SELECT *, {', '.join(f'pctl_{col} AS {short}' for short, col in LEGACY.items())} FROM read_parquet('{OUT_FP}'))
  TO '{OUT_FP}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD)""")
os.replace(OUT_FP + '.tmp', OUT_FP)
m = pq.ParquetFile(OUT_FP)
print(f'WROTE {OUT_FP}  ({m.metadata.num_rows:,} rows, {len(m.schema_arrow.names)} cols, {os.path.getsize(OUT_FP)/1e6:.0f} MB)')

## 2. Summary

In [ ]:
print(f'  {"column":<30}{"mean":>9}{"= min ties":>12}{"top 1%":>9}')
for col in ['C_3', 'C_5', 'C_10', 'C_all', 'uniqueC_5', 'uniqueC_all', 'uniqueC_examiner_5', 'uniqueC_applicant_5']:
    r = con.execute(f"""WITH x AS (SELECT pctl_{col} AS p, min(pctl_{col}) OVER (PARTITION BY wipo_sector, filing_year) AS mn FROM read_parquet('{OUT_FP}'))
                        SELECT avg(p), 100.0*avg((p = mn)::INT), 100.0*avg((p >= 0.99)::INT) FROM x""").fetchone()
    print(f'  {col:<30}{r[0]:>9.4f}{r[1]:>11.1f}%{r[2]:>8.2f}%')
print('\n  a large "= min ties" share is most applications receiving no citation of that kind at all')
display(con.execute(f"SELECT wipo_sector, count(*) AS n, count(DISTINCT filing_year) AS years FROM read_parquet('{OUT_FP}') GROUP BY 1 ORDER BY n DESC").fetchdf())
display(con.execute(f"SELECT appln_id, wipo_sector, filing_year, pctl_c5, pctl_call, pctl_uniqueC_5, pctl_uniqueC_all FROM read_parquet('{OUT_FP}') WHERE pctl_call > 0.99 LIMIT 6").fetchdf())
con.close()